In [3]:
%pip install numpy pandas scipy scikit-learn


[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### This approach involves only using audiologist recorded data and audiologist labels. Removing the layman data from the training. This notebook classifies the data as normal (pass) and abnormal (refer to a doctor).

### NOTE: this approach combines right and left ear data

In [4]:
import sys; print(sys.executable)


/usr/local/bin/python3


In [5]:
#jupyter command to draw plots on the notebook
%matplotlib inline

import numpy as np
import pandas as pd
import pickle               #saves object into file and loads it back
import math
import scipy.signal         # works with curves, peak finder
from scipy.stats import skew    # measures asymmetry of the probability distribution
from scipy.signal import butter, sosfiltfilt        #filtering
from sklearn.model_selection import GroupShuffleSplit     #creates training and validation splits
from collections import Counter     # checks class balance (how many samples per class)

#plt.rcParams['image.aspect'] = 'auto'
#plt.rcParams['font.size'] = 14

## Cleaning Data

In [6]:
zfile = np.load('../data/processed_data.zip')
data = pd.read_csv('../data/master_processed_no_duplicates_manual_ds.csv')

In [7]:
data.head()

,Difference_R,Difference_L,UIN,Year,Human_Type_R_audiologist,Human_Type_L_audiologist,Human_Type_R_layman,Human_Type_L_layman,Otoscopy - Right ear: (choice=Normal),Otoscopy - Right ear: (choice=Non-occluding cerumen),...,R_layman_manual_read,L_layman_manual_read,TPP_R_audiologist_manual,ECV_R_audiologist_manual,TPP_L_audiologist_manual,ECV_L_audiologist_manual,TPP_R_layman_manual,ECV_R_layman_manual,TPP_L_layman_manual,ECV_L_layman_manual
0,NaN,NaN,1010001,Year 1,A,A,A,A,False,False,...,True,True,-56.0,1.21,-9.0,1.14,-55.0,1.47,6.0,1.42
1,D,D,1010001,Year 2,C (<200 daPa),C (<200 daPa),B,B,False,False,...,True,True,-199.0,1.65,-224.0,1.21,NaN,1.60,-156.0,1.19
2,NaN,NaN,1010002,Year 1,A,A,A,A,False,True,...,True,True,10.0,0.81,1.0,1.09,16.0,1.04,10.0,1.32
3,NaN,NaN,1010002,Year 2,A,A,A,A,False,True,...,True,True,-14.0,1.27,-49.0,1.26,20.0,0.88,33.0,1.34
4,D,NaN,1010003,Year 2,A,A,B,A,False,False,...,True,True,-7.0,0.86,-32.0,0.98,NaN,0.05,-8.0,0.59


In [8]:
print(f'{len(np.unique(data["UIN"]))} unique UIN')

classes = ['A', 'B', 'C']
features = ['TPP', 'ECV', 'Type']

frames = []

# pulls R/L audiologist data into a single per-ear dataframe (audiologist only)
for ear in ['R', 'L']:
    names = ['UIN', f'Difference_{ear}']
    renames = ['UIN', 'Difference']

    for reader in ['audiologist']:
        names.append(f'Human_Type_{ear}_{reader}')
        renames.append(f'Human_Type_{reader}')

        names.append(f'Date_{reader}')
        renames.append(f'Date_{reader}')

        names.append(f'{ear}_test_idx_{reader}')
        renames.append(f'test_idx_{reader}')

        names.extend([f'{feat}_{ear}_{reader}' for feat in features])
        renames.extend([f'{feat}_{reader}' for feat in features])

        names.extend([f'{feat}_{ear}_{reader}_manual' for feat in features[:2]])
        renames.extend([f'{feat}_{reader}_manual' for feat in features[:2]])

    df = data[names].copy()
    df = df.rename(columns=dict(zip(names, renames)))

    for reader in ['audiologist']:
        def get_data_name(x):
            name = f'{x["UIN"]}_'+x[f"Date_{reader}"][:10]+f'_{reader}_{ear}'
            return name

        df[f'data_name_{reader}'] = df.apply(get_data_name, axis=1)
        df = df.drop(columns=f'Date_{reader}')

    frames.append(df)

frames = pd.concat(frames, ignore_index=True)
frames = frames.replace({
    'C  (<200 daPa)': 'C',
    'TYPE_A': 'A',
    'TYPE_AS': 'A',
    'TYPE_AD': 'A',
    'TYPE_B': 'B',
    'TYPE_C': 'C',
})

print(f'{len(frames)} intial tracings')

# Remove no test_idx  (audiologist only)
has_test_idx = frames['test_idx_audiologist'].notna()
print(f'{(~has_test_idx).sum()} ears without test index')
frames = frames[has_test_idx]
frames.reset_index(drop=True, inplace=True)

# Remove no machine type  (audiologist only)
has_machine_type = frames['Type_audiologist'].isin(classes)
print(f'{(~has_machine_type).sum()} ears without machine type')
frames = frames[has_machine_type]
frames.reset_index(drop=True, inplace=True)

# Remove CNE evals  (audiologist only)
has_human_type = frames['Human_Type_audiologist'].isin(classes)
print(f'{(~has_human_type).sum()} ears with non-CNE class')
frames = frames[has_human_type]
frames.reset_index(drop=True, inplace=True)

print(f'{len(np.unique(frames["UIN"]))} Unique UIN after removing bad entries')
print(f'{len(frames)} total ears')
print(f'{len(frames)} usable tracings')

# Check tracings are in zip file  (audiologist only)
datafile_names = [f[15:] for f in zfile.files[1:]]
assert frames['data_name_audiologist'].isin(datafile_names).all()

1583 unique UIN
4984 intial tracings
43 ears without test index
1 ears without machine type
9 ears with non-CNE class
1581 Unique UIN after removing bad entries
4931 total ears
4931 usable tracings


In [9]:
print(len(frames))
print(frames['data_name_audiologist'].str[-1].value_counts())

4931
data_name_audiologist
R    2470
L    2461
Name: count, dtype: int64


In [10]:
frames.head()

,UIN,Difference,Human_Type_audiologist,test_idx_audiologist,TPP_audiologist,ECV_audiologist,Type_audiologist,TPP_audiologist_manual,ECV_audiologist_manual,data_name_audiologist
0,1010001,NaN,A,2.0,-56.0,1.21,A,-56.0,1.21,1010001_2018-04-02_audiologist_R
1,1010001,D,C,0.0,-199.0,1.65,A,-199.0,1.65,1010001_2019-03-19_audiologist_R
2,1010002,NaN,A,0.0,5.0,0.79,A,10.0,0.81,1010002_2018-04-02_audiologist_R
3,1010002,NaN,A,0.0,-22.0,1.24,A,-14.0,1.27,1010002_2019-03-19_audiologist_R
4,1010003,D,A,0.0,-7.0,0.86,A,-7.0,0.86,1010003_2019-03-19_audiologist_R


# Split Dataset

In [11]:
'''
def count_labels(index_set, name, global_total=None):
    counts = Counter()
    for idx in index_set:
        label = frames.loc[idx].get('Human_Type_audiologist')   # audiologist only
        if label in ['A', 'B', 'C']:
            counts[label] += 1

    ordered = {k: counts.get(k, 0) for k in ['A', 'B', 'C']}
    subtotal = sum(ordered.values())

    print(f"\n{name} Set Label Counts:")
    for k in ['A', 'B', 'C']:
        count = ordered[k]
        pct = f"{(100 * count / global_total):.2f}%" if global_total else ""
        print(f"  {k}: {count:,} ({pct})")
    print(f"  Subtotal: {subtotal:,} ({(100 * subtotal / global_total):.2f}%)")
    '''

def count_labels(index_set, name, global_total=None):
    counts = Counter()
    for idx in index_set:
        label = frames.loc[idx].get('Human_Type_audiologist')
        if label == 'A':
            counts['normal'] += 1
        elif label in ['B', 'C']:
            counts['abnormal'] += 1

    ordered = {k: counts.get(k, 0) for k in ['normal', 'abnormal']}
    subtotal = sum(ordered.values())

    print(f"\n{name} Set Label Counts:")
    for k in ['normal', 'abnormal']:
        count = ordered[k]
        pct = f"{(100 * count / global_total):.2f}%" if global_total else ""
        print(f"  {k}: {count:,} ({pct})")
    print(f"  Subtotal: {subtotal:,} ({(100 * subtotal / global_total):.2f}%)")

In [12]:
# Seperate totoal dataset into a training set for neuton, a validation set for neuton, and a set to analyse the accuracy of the neuton model
splitter = GroupShuffleSplit(n_splits=1, train_size=0.9, test_size=0.1, random_state=42) # Use GroupShuffleSplit to ensure reproducibility
train_val_idx, analysis_idx = next(splitter.split(frames, groups=frames['UIN']))

frames_trainval = frames.iloc[train_val_idx].copy()
gss_inner = GroupShuffleSplit(n_splits=1, train_size=0.8, test_size=0.2, random_state=99)
train_idx_rel, val_idx_rel = next(gss_inner.split(frames_trainval, groups=frames_trainval['UIN']))

# Absolute indices
analysis_idx = frames.index[analysis_idx]
train_idx = frames_trainval.index[train_idx_rel]
val_idx = frames_trainval.index[val_idx_rel]

In [13]:
total_labeled = 0
for idx in frames.index:
    if frames.loc[idx, 'Human_Type_audiologist'] in ['A', 'B', 'C']:
        total_labeled += 1

count_labels(train_idx, "Training", global_total=total_labeled)
count_labels(val_idx, "Validation", global_total=total_labeled)
count_labels(analysis_idx, "Analysis", global_total=total_labeled)


Training Set Label Counts:
  normal: 3,179 (64.47%)
  abnormal: 371 (7.52%)
  Subtotal: 3,550 (71.99%)

Validation Set Label Counts:
  normal: 810 (16.43%)
  abnormal: 85 (1.72%)
  Subtotal: 895 (18.15%)

Analysis Set Label Counts:
  normal: 441 (8.94%)
  abnormal: 45 (0.91%)
  Subtotal: 486 (9.86%)


Formal Dataset for ML Training

In [14]:
def get_interp(trace):
    if trace[0][0] > trace[0][1]:
        y = np.interp(standard_x, trace[0][::-1], trace[1][::-1])
    elif trace[0][0] < trace[0][1]:
        raise RuntimeError

    y /= 100
    z = sosfiltfilt(lpf, y) #applies lpf
    return np.stack([y, z])

In [15]:
def extract_trace_features(trace):
    pressure = trace[0]
    compliance = trace[1]

    sa = float(np.max(compliance))
    sa_idx = int(np.argmax(compliance))
    est_tpp = float(pressure[sa_idx])
    est_ecv = compliance[-1]/100

    half_max = sa / 2
    over_half = np.where(compliance >= half_max)[0]
    tw = float(pressure[over_half[-1]] - pressure[over_half[0]] if len(over_half) else 0)

    return {
        "sa": sa,
        "est_tpp": est_tpp,
        "est_ecv": est_ecv,
        "tw": tw
    }

In [16]:
standard_x = np.arange(-399, 201, step=10, dtype=float)
lpf = butter(4, 0.04, 'lowpass', output='sos')

In [17]:
# Two class labels: 0 for normal, 1 for abnormal (B and C)
label_map = {'A': 0, 'B': 1, 'C': 1}

rows_train, rows_val, rows_analysis = [], [], []

train_set = set(train_idx)
val_set = set(val_idx)
analysis_set = set(analysis_idx)

for idx, row in frames.iterrows():
    label = row['Human_Type_audiologist']          # audiologist only, no reader loop
    if label not in label_map:
        continue

    try:
        uin = row['UIN']

        # by NAME, not position — manual values, NaN for flat curves
        tpp = row['TPP_audiologist_manual']
        ecv = row['ECV_audiologist_manual']

        no_peak = int(pd.isna(tpp))                # the flat-curve signature, as a feature

        raw_data = zfile['processed_data/' + row['data_name_audiologist'] + '.npy']
        raw_trace = np.stack([raw_data[0], raw_data[1]])
        trace = get_interp(raw_trace)
        trace_feats = extract_trace_features(raw_trace)

        # use the manual reading where it exists; for flat curves fall back to the estimate
        tpp_feat = trace_feats["est_tpp"] if pd.isna(tpp) else tpp
        ecv_feat = trace_feats["est_ecv"] if pd.isna(ecv) else ecv

        features = {
            "uin": uin,
            "idx": idx,
            "target": label_map[label],            # the answer column — was missing before
            "no_peak": no_peak,
            "tpp": tpp_feat,
            "ecv": ecv_feat,
        }
        features.update(trace_feats)
        features.update({f"sample_{i+1}": val for i, val in enumerate(trace[1])})

        # no peak gate — every labelled audiologist trace is kept (flat curves included)
        if idx in train_set:
            rows_train.append(features)
        elif idx in val_set:
            rows_val.append(features)
        elif idx in analysis_set:
            rows_analysis.append(features)

    except KeyError:
        continue

pd.DataFrame(rows_train).to_csv("../training_data/train_neuton_ao_b.csv", index=False)
pd.DataFrame(rows_val).to_csv("../training_data/val_neuton_ao_b.csv", index=False)
pd.DataFrame(rows_analysis).to_csv("../training_data/analysis_neuton_ao_b.csv", index=False)

print(f"Saved: train={len(rows_train)}, val={len(rows_val)}, analysis={len(rows_analysis)}")

Saved: train=3550, val=895, analysis=486
